# Tokenization

## Pre-Context

Large Language Models cannot process raw text directly because neural networks operate only on numerical data. Therefore, the first step in every LLM pipeline is to convert text into smaller units called **tokens**, which can then be mapped to numerical IDs and processed by the Transformer.

## What is Tokenization?

Tokenization is the process of splitting raw text into smaller units called **tokens**. A token can represent a character, a word, a subword, or a byte, depending on the tokenization algorithm.

For example,

**Input:**

```text
Alice gave Bob a book.
```

**Output:**

```text
["Alice", "gave", "Bob", "a", "book", "."]
```

These tokens are then converted into token IDs.

## Types of Tokenization

### 1. Character Tokenization

Splits text into individual characters.

Example:

```text
"cat"

↓

["c", "a", "t"]
```

### 2. Word Tokenization

Splits text into complete words.

Example:

```text
"The cat sat"

↓

["The", "cat", "sat"]
```

### 3. Subword Tokenization (Most Common)

Splits words into meaningful subwords.

Example:

```text
"unbelievable"

↓

["un", "believ", "able"]
```

It is widely used because it can represent unknown words while maintaining a manageable vocabulary size.

### 4. Byte-Level Tokenization

Converts text into byte representations before tokenization.

It can represent every Unicode character without using unknown tokens.

## Modern Tokenization Algorithms

Modern LLMs primarily use **Subword Tokenization** implemented through one of the following algorithms.

| Algorithm | Used By |
|------------|---------|
| Byte Pair Encoding (BPE) | GPT-2, GPT-3 |
| Byte-Level BPE | GPT-4, GPT-4o |
| SentencePiece | Llama, Mistral, Gemma, Qwen |
| WordPiece | BERT |

## Libraries and Frameworks

The most commonly used tokenization libraries are:

- **Hugging Face Transformers** (`AutoTokenizer`)
- **Hugging Face Tokenizers**
- **SentencePiece**
- **tiktoken** (OpenAI)

Example:

```python
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3-8B")

tokens = tokenizer.tokenize("Alice gave Bob a book.")
```

## Example

Input Text:

```text
Alice gave Bob a book.
```

Tokenization:

```text
["Alice", "gave", "Bob", "a", "book", "."]
```

Token IDs:

```text
[1523, 8432, 2941, 64, 5821, 13]
```

The token IDs are then passed to the **Embedding Layer**, where each ID is mapped to a dense vector representation.

## Pipeline

$$
\text{Raw Text}
\rightarrow
\text{Tokenization}
\rightarrow
\text{Tokens}
\rightarrow
\text{Token IDs}
\rightarrow
\text{Embedding Layer}
$$

# Embedding Layer

## Pre-Context

After tokenization, the input text has been converted into a sequence of token IDs. However, token IDs are simply integers and do not contain any semantic meaning. Therefore, each token ID is mapped to a dense numerical vector called an **embedding**, allowing the Transformer to capture semantic relationships between words.

## What is an Embedding?

An embedding is a dense vector representation of a token in a continuous vector space. Tokens with similar meanings tend to have similar embedding vectors, enabling the model to understand semantic relationships.

For example,

**Token IDs:**

```text
[1523, 8432, 2941, 64, 5821]
```

**Embedding Lookup:**

$$
E=
\begin{bmatrix}
0.2 & 0.8 & 0.1 & 0.5\\
0.9 & 0.3 & 0.4 & 0.1\\
0.5 & 0.2 & 0.7 & 0.3\\
0.6 & 0.9 & 0.2 & 0.4\\
0.3 & 0.4 & 0.8 & 0.6
\end{bmatrix}
$$

Each row corresponds to one token.

## Why are Embeddings Needed?

Token IDs have no mathematical meaning.

For example,

```text
Cat → 1523

Dog → 8432
```

The model cannot infer that **cat** and **dog** are semantically similar from these IDs alone.

Embeddings transform token IDs into dense vectors that preserve semantic relationships, allowing similar words to have similar vector representations.

## Types of Embeddings

### 1. One-Hot Encoding

Each token is represented by a vector containing a single 1 and the remaining values as 0.

Example:

$$
Cat=
[1,0,0,0]
$$

Not used in modern LLMs because the vectors are sparse and cannot capture semantic similarity.

### 2. Static Word Embeddings

Each word has one fixed embedding regardless of context.

Examples:

- Word2Vec
- GloVe
- FastText

### 3. Contextual Embeddings (Most Common)

The embedding of a word changes depending on its surrounding context.

Example:

The word **bank** in:

```text
I deposited money in the bank.
```

and

```text
The boat reached the river bank.
```

will eventually have different contextual representations after passing through the Transformer.

Modern LLMs use contextual embeddings.

## Weight Initialization

The embedding matrix is initialized with random values.

Example:

$$
E
\sim
\mathcal{N}(0,0.02)
$$

During training, backpropagation updates these values so that semantically similar words move closer together in the embedding space.

## Libraries and Frameworks

Common libraries used for embedding lookup include:

- **PyTorch** (`torch.nn.Embedding`)
- **TensorFlow** (`tf.keras.layers.Embedding`)
- **Hugging Face Transformers**
- **SentenceTransformers** (for sentence embeddings)

Example:

```python
import torch.nn as nn

embedding = nn.Embedding(
    num_embeddings=32000,
    embedding_dim=4096
)
```

## Example

Input Token IDs:

```text
[1523, 8432, 2941, 64]
```

Embedding Matrix:

$$
E=
\begin{bmatrix}
0.2 & 0.8 & 0.1 & 0.5\\
0.9 & 0.3 & 0.4 & 0.1\\
0.5 & 0.2 & 0.7 & 0.3\\
0.6 & 0.9 & 0.2 & 0.4
\end{bmatrix}
$$

Shape:

$$
E
\in
\mathbb{R}^{sequence\ length \times embedding\ dimension}
$$

For this example,

$$
E
\in
\mathbb{R}^{4\times4}
$$

The embedding matrix becomes the input to the **Positional Encoding** stage.

## Pipeline

$$
\text{Raw Text}
\rightarrow
\text{Tokenization}
\rightarrow
\text{Token IDs}
\rightarrow
\text{Embedding Lookup}
\rightarrow
\text{Embedding Matrix}
\rightarrow
\text{Positional Encoding}
$$

# Positional Encoding

## Pre-Context

After the embedding layer, every token is represented as a dense vector containing semantic information. However, embeddings do not preserve the order of tokens in a sentence. Since the Transformer processes all tokens in parallel, it has no inherent understanding of sequence order. Positional Encoding is introduced to inject position information into the token embeddings.

## What is Positional Encoding?

Positional Encoding is a technique used to provide positional information to the Transformer. It enables the model to distinguish between tokens based on their position within a sequence while preserving their semantic meaning.

For example,

Input Sentence:

```text
The cat sat on the mat
```

Without positional information, the model would treat the sentence as an unordered collection of tokens.

With positional encoding, each embedding contains both:

- Semantic Information
- Position Information

The position-aware embedding is computed as:

$$
X_{pos}=X+P
$$

where

- \(X\) is the embedding matrix.
- \(P\) is the positional encoding matrix.

## Why is Positional Encoding Needed?

Without positional encoding:

- The Transformer cannot determine which token appears first.
- Word order is completely lost.
- Different sentences containing the same words may produce identical embeddings.

For example,

```text
Dog bites man
```

and

```text
Man bites dog
```

contain the same words but have completely different meanings.

Positional Encoding enables the model to distinguish between these sequences.

## Types of Positional Encoding

### 1. Fixed Sinusoidal Encoding

Introduced in the original **Attention Is All You Need (2017)** paper.

The positional values are computed using sine and cosine functions.

$$
PE(pos,2i)
=
\sin
\left(
\frac{pos}
{10000^{2i/d_{model}}}
\right)
$$

$$
PE(pos,2i+1)
=
\cos
\left(
\frac{pos}
{10000^{2i/d_{model}}}
\right)
$$

**Used in:**

- Original Transformer



### 2. Learned Positional Embeddings

Instead of using mathematical functions, the positional vectors are initialized randomly and learned during training using backpropagation.

**Used in:**

- GPT-2
- BERT



### 3. Rotary Position Embedding (RoPE) 

Modern Large Language Models no longer add positional vectors directly to embeddings.

Instead, positional information is applied by rotating the **Query** and **Key** vectors before computing attention.

RoPE preserves relative positional information and enables better generalization to longer sequences.

**Used in:**

- Llama
- Mistral
- Gemma
- Qwen
- DeepSeek

## Modern Method

The most widely used positional encoding method in modern LLMs is **Rotary Position Embedding (RoPE)** because it provides better long-context performance and improves relative position modeling.

## Libraries and Frameworks

Common frameworks supporting positional encoding include:

- **PyTorch**
- **TensorFlow**
- **Hugging Face Transformers**
- **xFormers**
- **FlashAttention**

## Example

Suppose the embedding matrix is:

$$
X=
\begin{bmatrix}
0.2 & 0.8 & 0.1 & 0.5 \\
0.9 & 0.3 & 0.4 & 0.1 \\
0.5 & 0.2 & 0.7 & 0.3 \\
0.6 & 0.9 & 0.2 & 0.4
\end{bmatrix}
$$

Assume the positional encoding matrix is:

$$
P=
\begin{bmatrix}
0.1 & 0.0 & 0.0 & 0.0 \\
0.0 & 0.1 & 0.0 & 0.0 \\
0.0 & 0.0 & 0.1 & 0.0 \\
0.0 & 0.0 & 0.0 & 0.1
\end{bmatrix}
$$

The position-aware embedding matrix is computed as:

$$
X_{pos}=X+P
$$

Result:

$$
X_{pos}=
\begin{bmatrix}
0.3 & 0.8 & 0.1 & 0.5 \\
0.9 & 0.4 & 0.4 & 0.1 \\
0.5 & 0.2 & 0.8 & 0.3 \\
0.6 & 0.9 & 0.2 & 0.5
\end{bmatrix}
$$

The resulting matrix now contains both semantic and positional information and becomes the input to the normalization layer in modern Transformer architectures.

## Pipeline

$$
\text{Raw Text}
\rightarrow
\text{Tokenization}
\rightarrow
\text{Token IDs}
\rightarrow
\text{Embedding Layer}
\rightarrow
\text{Positional Encoding}
\rightarrow
\text{Normalization}
$$

# Normalization

## Pre-Context

After positional information is incorporated into the embeddings, the resulting vectors are passed through a normalization layer before being used to generate the Query, Key, and Value matrices. Normalization stabilizes the numerical values of the embeddings, leading to faster convergence and more stable training of deep Transformer models.

## What is Normalization?

Normalization is the process of scaling the input features so that they have a stable numerical distribution. This prevents activations from becoming extremely large or extremely small as they propagate through multiple Transformer layers.

Without normalization, deep neural networks may suffer from unstable gradients, making optimization difficult.

## Why is Normalization Needed?

Normalization provides several advantages:

- Stabilizes the training process.
- Prevents exploding and vanishing gradients.
- Improves gradient flow during backpropagation.
- Enables deeper Transformer architectures.
- Speeds up model convergence.
- Improves numerical stability.

## Types of Normalization

### 1. Batch Normalization

Batch Normalization computes statistics across an entire mini-batch.

It is widely used in Convolutional Neural Networks (CNNs) but is generally not suitable for Transformers because sequence lengths vary and inference often processes one sample at a time.

**Used in:**

- CNNs
- ResNet
- EfficientNet



### 2. Layer Normalization

Layer Normalization computes the mean and variance across the features of a single token independently.

For an input vector:

$$
x=[x_1,x_2,\ldots,x_n]
$$

the normalized output is computed as:

$$
\mu=\frac{1}{n}\sum_{i=1}^{n}x_i
$$

$$
\sigma^2=\frac{1}{n}\sum_{i=1}^{n}(x_i-\mu)^2
$$

$$
LayerNorm(x)=
\gamma
\frac{x-\mu}{\sqrt{\sigma^2+\epsilon}}
+\beta
$$

where:

- $\gamma$ is the learnable scale parameter.
- $\beta$ is the learnable bias parameter.
- $\epsilon$ is a small constant for numerical stability.

**Used in:**

- Original Transformer
- BERT
- GPT-2



### 3. RMS Normalization (RMSNorm)

Modern Large Language Models replace LayerNorm with RMSNorm.

Instead of computing both the mean and variance, RMSNorm computes only the Root Mean Square (RMS).

$$
RMS(x)
=
\sqrt{
\frac{1}{n}
\sum_{i=1}^{n}
x_i^2
}
$$

The normalized output becomes:

$$
RMSNorm(x)
=
\gamma
\frac{x}
{RMS(x)+\epsilon}
$$

Unlike LayerNorm, RMSNorm does not subtract the mean, making it computationally simpler and more efficient.

**Used in:**

- Llama
- Mistral
- Gemma
- Qwen
- DeepSeek

## Modern Method

Modern decoder-only Large Language Models primarily use **RMSNorm** because it provides similar performance to LayerNorm while requiring fewer computations.

## Libraries and Frameworks

The most commonly used implementations are:

- **PyTorch** (`torch.nn.LayerNorm`)
- **PyTorch** (`torch.nn.RMSNorm`)
- **TensorFlow**
- **Hugging Face Transformers**

Example:

```python
import torch.nn as nn

layer_norm = nn.LayerNorm(4096)

rms_norm = nn.RMSNorm(4096)
```

## Example

Suppose the input embedding is:

$$
x=
[1.2,\ 0.8,\ 1.5,\ 0.9]
$$

After normalization:

$$
x_{norm}
=
[0.73,\ -0.41,\ 1.28,\ -0.32]
$$

The normalized vector is then passed to the next stage of the Transformer.

## Pipeline

$$
\text{Raw Text}
\rightarrow
\text{Tokenization}
\rightarrow
\text{Token IDs}
\rightarrow
\text{Embedding Layer}
\rightarrow
\text{Positional Encoding}
\rightarrow
\text{Normalization}
\rightarrow
\text{Query, Key, Value Generation}
$$

# Query, Key, and Value (QKV) Generation

## Pre-Context

After normalization, the input embeddings contain both semantic and positional information in a stable numerical form. However, the Transformer still cannot determine which tokens should attend to one another. To enable this, the normalized embeddings are projected into three different vector spaces called **Query (Q)**, **Key (K)**, and **Value (V)**.

## What are Query, Key, and Value?

Query, Key, and Value are three different representations of the same input embedding obtained by multiplying the normalized input with three learnable weight matrices.

Each token produces:

- A **Query (Q)** vector to ask what information it needs.
- A **Key (K)** vector to indicate what information it contains.
- A **Value (V)** vector that stores the actual information to be shared.

These vectors are generated using linear transformations.

$$
Q=X_{norm}W_Q
$$

$$
K=X_{norm}W_K
$$

$$
V=X_{norm}W_V
$$

where

- $X_{norm}$ is the normalized input matrix.
- $W_Q$ is the Query weight matrix.
- $W_K$ is the Key weight matrix.
- $W_V$ is the Value weight matrix.

## Why are Q, K, and V Needed?

Embeddings alone cannot determine relationships between different tokens.

The QKV mechanism enables the model to:

- Compare tokens.
- Measure similarity.
- Compute attention scores.
- Exchange contextual information.

Without QKV, the self-attention mechanism cannot operate.

## Weight Initialization

The Query, Key, and Value matrices are initialized randomly before training.

Common initialization methods include:

- Xavier Initialization
- Kaiming Initialization
- Normal Initialization

During training, these matrices are updated using backpropagation.

## Modern Method

All modern Transformer models use learnable linear projection layers to generate Query, Key, and Value vectors.

Examples:

- GPT-4
- Llama
- Mistral
- Gemma
- Qwen
- DeepSeek

## Libraries and Frameworks

Common implementations include:

- **PyTorch** (`torch.nn.Linear`)
- **TensorFlow** (`tf.keras.layers.Dense`)
- **Hugging Face Transformers**

Example:

```python
import torch.nn as nn

W_Q = nn.Linear(4096,4096)
W_K = nn.Linear(4096,4096)
W_V = nn.Linear(4096,4096)
```

## Example

Suppose the normalized input matrix is:

$$
X_{norm}=
\begin{bmatrix}
0.3 & 0.8 & 0.1 & 0.5 \\
0.9 & 0.4 & 0.4 & 0.1 \\
0.5 & 0.2 & 0.8 & 0.3 \\
0.6 & 0.9 & 0.2 & 0.5
\end{bmatrix}
$$

Assume the learnable weight matrices are:

$$
W_Q=
\begin{bmatrix}
1 & 0 \\
0 & 1 \\
1 & 0 \\
0 & 1
\end{bmatrix}
$$

$$
W_K=
\begin{bmatrix}
1 & 1 \\
0 & 1 \\
1 & 0 \\
0 & 1
\end{bmatrix}
$$

$$
W_V=
\begin{bmatrix}
1 & 0 \\
1 & 1 \\
0 & 1 \\
1 & 0
\end{bmatrix}
$$

The Query matrix is computed as:

$$
Q=X_{norm}W_Q
$$

The Key matrix is computed as:

$$
K=X_{norm}W_K
$$

The Value matrix is computed as:

$$
V=X_{norm}W_V
$$

These matrices become the input to the Self-Attention mechanism.

## Pipeline

$$
\text{Raw Text}
\rightarrow
\text{Tokenization}
\rightarrow
\text{Token IDs}
\rightarrow
\text{Embedding Layer}
\rightarrow
\text{Positional Encoding}
\rightarrow
\text{Normalization}
\rightarrow
\text{QKV Generation}
\rightarrow
\text{Self-Attention}
$$

# Self-Attention

## Pre-Context

After generating the Query (Q), Key (K), and Value (V) matrices, the Transformer has separate representations describing what each token is looking for, what information each token contains, and what information can be shared. The next step is to determine how strongly every token should attend to every other token. This process is performed by the **Self-Attention** mechanism.

## What is Self-Attention?

Self-Attention is the mechanism that enables each token in a sequence to attend to all other tokens and gather the most relevant contextual information.

Instead of processing each token independently, Self-Attention allows every token to learn relationships with every other token in the same sequence.

## Why is Self-Attention Needed?

Without Self-Attention:

- Tokens cannot communicate with one another.
- The model only understands individual token meanings.
- Long-range dependencies cannot be captured.

Self-Attention enables the model to learn contextual relationships between tokens regardless of their distance in the sequence.

## Steps in Self-Attention

### 1. Compute Attention Scores

The similarity between Query and Key vectors is computed using a dot product.

$$
Scores = QK^T
$$

Each element represents how strongly one token attends to another.



### 2. Scale the Scores

Large dot-product values can make the Softmax function unstable.

Therefore, the scores are scaled by:

$$
\frac{QK^T}{\sqrt{d_k}}
$$

where:

- $d_k$ is the dimension of the Key vectors.



### 3. Apply Softmax

The scaled scores are converted into probabilities.

$$
A=
Softmax
\left(
\frac{QK^T}{\sqrt{d_k}}
\right)
$$

Each row of the attention matrix sums to 1.



### 4. Compute the Weighted Sum

The attention weights are multiplied by the Value matrix.

$$
Output=A\times V
$$

or

$$
Attention(Q,K,V)
=
Softmax
\left(
\frac{QK^T}{\sqrt{d_k}}
\right)V
$$

The resulting vectors are called **contextual embeddings**.

## Why Scale the Attention Scores?

Without scaling, the dot-product values become large as the embedding dimension increases.

Large values produce extremely peaked Softmax outputs, leading to unstable gradients.

Scaling improves numerical stability during training.

## Modern Method

All modern Transformer-based LLMs use **Scaled Dot-Product Self-Attention**.

Examples:

- GPT
- Llama
- Mistral
- Gemma
- Qwen
- DeepSeek

## Libraries and Frameworks

Common implementations include:

- **PyTorch** (`torch.nn.MultiheadAttention`)
- **PyTorch** (`torch.nn.functional.scaled_dot_product_attention`)
- **FlashAttention**
- **Hugging Face Transformers**

Example:

```python
import torch.nn.functional as F

output = F.scaled_dot_product_attention(Q, K, V)
```

## Example

Suppose

$$
Q=
\begin{bmatrix}
0.4 & 1.3\\
1.3 & 0.5\\
1.3 & 0.5\\
0.8 & 1.4
\end{bmatrix}
$$

$$
K=
\begin{bmatrix}
0.4 & 1.7\\
1.3 & 1.8\\
1.3 & 1.0\\
0.8 & 2.3
\end{bmatrix}
$$

$$
V=
\begin{bmatrix}
1.6 & 0.9\\
1.4 & 0.8\\
1.0 & 1.0\\
2.0 & 1.1
\end{bmatrix}
$$

The attention scores are computed as:

$$
Scores=QK^T
$$

The scores are then scaled:

$$
Scaled=
\frac{QK^T}{\sqrt{d_k}}
$$

Next, Softmax is applied:

$$
A=
Softmax(Scaled)
$$

Finally,

$$
Output=A\times V
$$

The output matrix now contains contextual representations for every token.

## Pipeline

$$
\text{Raw Text}
\rightarrow
\text{Tokenization}
\rightarrow
\text{Token IDs}
\rightarrow
\text{Embedding Layer}
\rightarrow
\text{Positional Encoding}
\rightarrow
\text{Normalization}
\rightarrow
\text{QKV Generation}
\rightarrow
\text{Self-Attention}
\rightarrow
\text{Multi-Head Attention}
$$

# Multi-Head Attention

## Pre-Context

The Self-Attention mechanism enables each token to learn contextual information from all other tokens in the sequence. However, using only a single attention head limits the model to learning one type of relationship at a time. Modern Transformers overcome this limitation by employing **Multi-Head Attention**, where multiple attention heads learn different semantic and syntactic relationships simultaneously.

## What is Multi-Head Attention?

Multi-Head Attention is an extension of the Self-Attention mechanism in which several independent attention heads operate in parallel. Each attention head has its own Query, Key, and Value projection matrices and learns different contextual relationships.

Instead of computing attention once, the Transformer computes it multiple times in parallel.

## Why is Multi-Head Attention Needed?

Using a single attention head forces the model to learn only one representation of the input sequence.

Multi-Head Attention enables the model to learn multiple relationships simultaneously, such as:

- Semantic relationships
- Syntactic relationships
- Long-range dependencies
- Local dependencies
- Entity relationships

This improves the expressive power of the Transformer.

## Architecture

Suppose there are \(h\) attention heads.

For each attention head:

$$
Q_i=XW_Q^{(i)}
$$

$$
K_i=XW_K^{(i)}
$$

$$
V_i=XW_V^{(i)}
$$

Each head independently computes:

$$
head_i=
Attention(Q_i,K_i,V_i)
$$

All attention heads are then concatenated:

$$
H=
Concat(head_1,head_2,\ldots,head_h)
$$

Finally, the concatenated output is projected back to the original hidden dimension using the output projection matrix:

$$
Output=HW_O
$$

where

- \(H\) is the concatenated attention output.
- \(W_O\) is the learnable output projection matrix.

## Why Concatenate Multiple Heads?

Each attention head learns different information.

For example,

- Head 1 may learn grammatical relationships.
- Head 2 may learn subject-object relationships.
- Head 3 may focus on nearby words.
- Head 4 may capture long-range dependencies.

Concatenating all heads combines these learned representations into a single contextual embedding.

## Modern Method

All modern Transformer-based Large Language Models use Multi-Head Attention.

Examples:

- GPT
- Llama
- Mistral
- Gemma
- Qwen
- DeepSeek

## Libraries and Frameworks

Common implementations include:

- **PyTorch** (`torch.nn.MultiheadAttention`)
- **PyTorch** (`torch.nn.functional.scaled_dot_product_attention`)
- **FlashAttention**
- **Hugging Face Transformers**

Example:

```python
import torch.nn as nn

mha = nn.MultiheadAttention(
    embed_dim=4096,
    num_heads=32,
    batch_first=True
)
```

## Example

Suppose the Transformer uses four attention heads.

Each head computes:

$$
head_1
=
Attention(Q_1,K_1,V_1)
$$

$$
head_2
=
Attention(Q_2,K_2,V_2)
$$

$$
head_3
=
Attention(Q_3,K_3,V_3)
$$

$$
head_4
=
Attention(Q_4,K_4,V_4)
$$

The outputs are concatenated:

$$
H=
Concat(head_1,head_2,head_3,head_4)
$$

Finally,

$$
Output=HW_O
$$

The resulting matrix becomes the output of the Multi-Head Attention layer.

## Pipeline

$$
\text{Raw Text}
\rightarrow
\text{Tokenization}
\rightarrow
\text{Token IDs}
\rightarrow
\text{Embedding Layer}
\rightarrow
\text{Positional Encoding}
\rightarrow
\text{Normalization}
\rightarrow
\text{QKV Generation}
\rightarrow
\text{Self-Attention}
\rightarrow
\text{Multi-Head Attention}
\rightarrow
\text{Output Projection}
$$

# Output Projection

## Pre-Context

After Multi-Head Attention, each attention head has learned different contextual relationships from the input sequence. However, these outputs are still separated into multiple heads. Before passing the information to the next layer, the Transformer combines all attention heads into a single representation using an **Output Projection** layer.

## What is Output Projection?

Output Projection is a learnable linear transformation that combines the outputs of all attention heads into a single hidden representation with the original embedding dimension.

Suppose there are \(h\) attention heads.

After concatenation, the output is:

$$
H=
Concat(head_1,head_2,\ldots,head_h)
$$

The concatenated matrix is then multiplied by a learnable weight matrix:

$$
Output=HW_O
$$

where

- \(H\) is the concatenated attention output.
- \(W_O\) is the Output Projection matrix.

## Why is Output Projection Needed?

The outputs of all attention heads are concatenated, increasing the feature dimension.

For example,

Suppose

- Embedding Dimension = 512
- Number of Heads = 8

Each attention head produces:

$$
64
$$

features.

After concatenation:

$$
8\times64=512
$$

The Output Projection learns how to combine these different representations into a single meaningful embedding.

Without Output Projection:

- Attention heads remain independent.
- Information from different heads cannot be integrated.
- The model cannot effectively combine different learned relationships.

## Weight Matrix

The Output Projection uses a learnable weight matrix:

$$
W_O
\in
\mathbb{R}^{d_{model}\times d_{model}}
$$

Like all other weight matrices, \(W_O\) is initialized randomly and updated during backpropagation.

## Modern Method

Every modern Transformer-based LLM includes an Output Projection layer.

Examples:

- GPT
- Llama
- Mistral
- Gemma
- Qwen
- DeepSeek

## Libraries and Frameworks

Common implementations include:

- **PyTorch** (`torch.nn.Linear`)
- **TensorFlow**
- **Hugging Face Transformers**

Example:

```python
import torch.nn as nn

output_projection = nn.Linear(
    in_features=4096,
    out_features=4096
)
```

## Example

Suppose two attention heads produce:

$$
head_1=
\begin{bmatrix}
0.2 & 0.5\\
0.7 & 0.1
\end{bmatrix}
$$

$$
head_2=
\begin{bmatrix}
0.4 & 0.8\\
0.2 & 0.9
\end{bmatrix}
$$

After concatenation:

$$
H=
\begin{bmatrix}
0.2 & 0.5 & 0.4 & 0.8\\
0.7 & 0.1 & 0.2 & 0.9
\end{bmatrix}
$$

The Transformer computes:

$$
Output=HW_O
$$

The resulting matrix has the same embedding dimension as the original input and becomes the input to the Residual Connection.

## Pipeline

$$
\text{Raw Text}
\rightarrow
\text{Tokenization}
\rightarrow
\text{Token IDs}
\rightarrow
\text{Embedding Layer}
\rightarrow
\text{Positional Encoding}
\rightarrow
\text{Normalization}
\rightarrow
\text{QKV Generation}
\rightarrow
\text{Self-Attention}
\rightarrow
\text{Multi-Head Attention}
\rightarrow
\text{Output Projection}
\rightarrow
\text{Residual Connection}
$$

# Residual Connection

## Pre-Context

After the Multi-Head Attention outputs have been combined through the Output Projection layer, the Transformer does not directly pass this output to the next layer. Instead, it adds the original input back to the projected output using a **Residual Connection**. This preserves the original information while allowing the model to learn only the additional contextual information.

## What is a Residual Connection?

A Residual Connection, also known as a **Skip Connection**, is an element-wise addition between the input of a layer and its output.

If the input to the Multi-Head Attention layer is

$$
X
$$

and the output after projection is

$$
O
$$

then the residual output is computed as

$$
Y=X+O
$$

Both matrices must have the same dimensions.

## Why is a Residual Connection Needed?

Residual Connections provide several important advantages:

- Preserve the original token representations.
- Improve gradient flow during backpropagation.
- Prevent vanishing gradients.
- Enable very deep Transformer architectures.
- Allow each layer to learn only the residual information instead of relearning the entire representation.

Without Residual Connections, training very deep neural networks becomes difficult.

## Mathematical Formulation

Let

$$
X
\in
\mathbb{R}^{n\times d_{model}}
$$

be the input matrix and

$$
O
\in
\mathbb{R}^{n\times d_{model}}
$$

be the output from the Multi-Head Attention layer.

The residual output is

$$
Y=X+O
$$

where the addition is performed element-wise.

## Modern Method

Residual Connections are used in all modern Transformer-based architectures.

Examples:

- GPT
- Llama
- Mistral
- Gemma
- Qwen
- DeepSeek
- BERT

Every Transformer block contains **two Residual Connections**:

1. After Multi-Head Attention.
2. After the Feed Forward Network.

## Libraries and Frameworks

Residual Connections are implemented using simple tensor addition.

Common frameworks include:

- **PyTorch**
- **TensorFlow**
- **JAX**

Example:

```python
output = x + attention_output
```

## Example

Suppose the input matrix is

$$
X=
\begin{bmatrix}
0.3 & 0.8\\
0.5 & 0.2
\end{bmatrix}
$$

and the Output Projection produces

$$
O=
\begin{bmatrix}
0.2 & 0.1\\
0.4 & 0.6
\end{bmatrix}
$$

The Residual Connection computes

$$
Y=X+O
$$

Result:

$$
Y=
\begin{bmatrix}
0.5 & 0.9\\
0.9 & 0.8
\end{bmatrix}
$$

The resulting matrix is passed to the next normalization layer.

## Pipeline

$$
\text{Raw Text}
\rightarrow
\text{Tokenization}
\rightarrow
\text{Token IDs}
\rightarrow
\text{Embedding Layer}
\rightarrow
\text{Positional Encoding}
\rightarrow
\text{Normalization}
\rightarrow
\text{QKV Generation}
\rightarrow
\text{Self-Attention}
\rightarrow
\text{Multi-Head Attention}
\rightarrow
\text{Output Projection}
\rightarrow
\text{Residual Connection}
\rightarrow
\text{Normalization}
$$

# Second Normalization (RMSNorm)

## Pre-Context

After the Residual Connection, the output contains both the original token representations and the contextual information learned through Multi-Head Attention. Before passing this representation to the Feed Forward Network, modern Transformers apply a second normalization layer. This normalization stabilizes the activations and ensures consistent numerical scaling before further feature transformation.

## What is Second Normalization?

The second normalization layer applies the same normalization technique used before the attention mechanism.

For modern Large Language Models, this is typically **RMSNorm**.

If the output of the Residual Connection is

$$
Y=X+O
$$

then the normalized output is

$$
Y_{norm}=RMSNorm(Y)
$$

where

- \(Y\) is the residual output.
- \(Y_{norm}\) is the normalized representation.

## Why is Second Normalization Needed?

The Residual Connection increases the magnitude of activations because two matrices are added together.

Without normalization:

- Activations may become unstable.
- Gradient flow becomes more difficult.
- Training deep Transformers becomes less efficient.

Applying RMSNorm before the Feed Forward Network maintains numerical stability throughout the model.

## Mathematical Formulation

For an input vector

$$
x=[x_1,x_2,\ldots,x_n]
$$

the Root Mean Square is

$$
RMS(x)
=
\sqrt{
\frac{1}{n}
\sum_{i=1}^{n}
x_i^2
}
$$

The normalized output is

$$
RMSNorm(x)
=
\gamma
\frac{x}
{RMS(x)+\epsilon}
$$

where

- \(\gamma\) is a learnable scaling parameter.
- \(\epsilon\) is a small constant for numerical stability.

## Modern Method

Modern decoder-only Large Language Models primarily use **RMSNorm**.

Examples:

- Llama
- Mistral
- Gemma
- Qwen
- DeepSeek

Older Transformer models such as BERT and GPT-2 use **LayerNorm** instead.

## Libraries and Frameworks

Common implementations include:

- **PyTorch** (`torch.nn.RMSNorm`)
- **TensorFlow**
- **Hugging Face Transformers**

Example:

```python
import torch.nn as nn

norm = nn.RMSNorm(4096)

output = norm(hidden_states)
```

## Example

Suppose the residual output is

$$
Y=
\begin{bmatrix}
1.05 & 1.42 & 0.22 & 0.98\\
1.68 & 1.04 & 0.79 & 0.25
\end{bmatrix}
$$

After applying RMSNorm:

$$
Y_{norm}
=
\begin{bmatrix}
0.72 & 0.98 & 0.15 & 0.68\\
1.01 & 0.63 & 0.48 & 0.15
\end{bmatrix}
$$

The normalized matrix is then passed to the Feed Forward Network.

## Pipeline

$$
\text{Raw Text}
\rightarrow
\text{Tokenization}
\rightarrow
\text{Token IDs}
\rightarrow
\text{Embedding Layer}
\rightarrow
\text{Positional Encoding}
\rightarrow
\text{RMSNorm}
\rightarrow
\text{Multi-Head Attention}
\rightarrow
\text{Output Projection}
\rightarrow
\text{Residual Connection}
\rightarrow
\text{RMSNorm}
\rightarrow
\text{Feed Forward Network}
$$

# Feed Forward Network (FFN)

## Pre-Context

After the second normalization, each token has already gathered contextual information through the Multi-Head Attention mechanism. However, attention only enables communication between tokens; it does not perform complex feature transformation. The Feed Forward Network (FFN) independently processes each token to learn richer and more abstract feature representations before passing them to the next Transformer layer.

## What is a Feed Forward Network?

A Feed Forward Network (FFN), also called a **Multi-Layer Perceptron (MLP)**, is a two-layer fully connected neural network applied independently to every token in the sequence.

Unlike Self-Attention, the FFN does not allow tokens to communicate with each other. Instead, it transforms each contextual embedding individually.

## Why is the Feed Forward Network Needed?

The attention mechanism gathers contextual information but performs only weighted combinations of Value vectors.

The Feed Forward Network:

- Learns higher-level feature representations.
- Introduces non-linearity.
- Increases the model's expressive power.
- Refines contextual embeddings.
- Prepares features for the next Transformer layer.

## Mathematical Formulation

The original Transformer uses:

$$
FFN(x)
=
W_2
\left(
ReLU(W_1x+b_1)
\right)
+b_2
$$

Modern Large Language Models commonly use:

$$
FFN(x)
=
W_2
\left(
GELU(W_1x+b_1)
\right)
+b_2
$$

Llama models replace GELU with SwiGLU:

$$
FFN(x)
=
W_2
\left(
SwiGLU(W_1x)
\right)
$$

where

- \(W_1\) expands the hidden dimension.
- The activation function introduces non-linearity.
- \(W_2\) projects the representation back to the original hidden dimension.

## Hidden Dimension Expansion

Suppose

$$
d_{model}=4096
$$

The FFN first expands the representation:

$$
4096
\rightarrow
11008
$$

Then projects it back:

$$
11008
\rightarrow
4096
$$

This allows the network to learn richer feature representations.

## Modern Method

Modern LLMs typically use **SwiGLU** because it improves performance over ReLU and GELU.

Examples:

- Llama
- Mistral
- Gemma
- Qwen
- DeepSeek

Older models such as GPT-2 and BERT use **GELU**.

## Libraries and Frameworks

Common implementations include:

- **PyTorch** (`torch.nn.Linear`)
- **TensorFlow**
- **Hugging Face Transformers**

Example:

```python
import torch.nn as nn

fc1 = nn.Linear(4096,11008)

fc2 = nn.Linear(11008,4096)
```

## Example

Suppose the normalized token embedding is

$$
x=
[1.48,\ 0.94]
$$

After the first linear layer:

$$
W_1x
=
[2.31,\ -0.84,\ 1.56,\ 0.42]
$$

After applying GELU:

$$
[2.28,\ -0.17,\ 1.47,\ 0.38]
$$

Finally,

$$
FFN(x)
=
[1.92,\ 1.14]
$$

This transformed vector becomes the input to the second Residual Connection.

## Pipeline

$$
\text{Raw Text}
\rightarrow
\text{Tokenization}
\rightarrow
\text{Token IDs}
\rightarrow
\text{Embedding Layer}
\rightarrow
\text{Positional Encoding}
\rightarrow
\text{RMSNorm}
\rightarrow
\text{Multi-Head Attention}
\rightarrow
\text{Output Projection}
\rightarrow
\text{Residual Connection}
\rightarrow
\text{RMSNorm}
\rightarrow
\text{Feed Forward Network}
\rightarrow
\text{Residual Connection}
$$

# Second Residual Connection

## Pre-Context

After the Feed Forward Network (FFN), each token has been transformed into a richer feature representation. However, similar to the attention layer, directly passing the FFN output to the next layer may cause the model to lose important information from earlier stages. Therefore, the Transformer applies a second Residual Connection, adding the FFN output back to its input.

## What is the Second Residual Connection?

The second Residual Connection performs an element-wise addition between the input of the Feed Forward Network and its output.

If

$$
Y
$$

is the input to the FFN, and

$$
FFN(Y)
$$

is the transformed output, then

$$
Z
=
Y
+
FFN(Y)
$$

where

- \(Y\) is the normalized input to the FFN.
- \(FFN(Y)\) is the transformed representation.
- \(Z\) is the final output of the Transformer block.

## Why is the Second Residual Connection Needed?

The second Residual Connection:

- Preserves information learned from the attention mechanism.
- Prevents degradation of deep networks.
- Improves gradient flow.
- Stabilizes optimization.
- Enables the FFN to learn only additional useful features.

Like the first Residual Connection, it helps the model learn the residual information instead of relearning the complete representation.

## Mathematical Formulation

The output is computed as

$$
Z
=
Y
+
FFN(Y)
$$

where the addition is performed element-wise.

Both matrices must have identical dimensions.

## Modern Method

Every modern Transformer-based LLM uses a second Residual Connection.

Examples:

- GPT
- Llama
- Mistral
- Gemma
- Qwen
- DeepSeek

Every Transformer block therefore contains:

- Residual Connection after Multi-Head Attention.
- Residual Connection after Feed Forward Network.

## Libraries and Frameworks

Residual Connections are implemented using simple tensor addition.

Common frameworks include:

- **PyTorch**
- **TensorFlow**
- **JAX**

Example:

```python
output = hidden_states + ffn_output
```

## Example

Suppose the FFN input is

$$
Y=
\begin{bmatrix}
1.05 & 1.42\\
0.98 & 0.74
\end{bmatrix}
$$

The Feed Forward Network produces

$$
FFN(Y)=
\begin{bmatrix}
0.32 & 0.18\\
0.25 & 0.40
\end{bmatrix}
$$

The second Residual Connection computes

$$
Z=
Y+FFN(Y)
$$

Result:

$$
Z=
\begin{bmatrix}
1.37 & 1.60\\
1.23 & 1.14
\end{bmatrix}
$$

The resulting matrix becomes the output of one complete Transformer block.

## Pipeline

$$
\text{Raw Text}
\rightarrow
\text{Tokenization}
\rightarrow
\text{Token IDs}
\rightarrow
\text{Embedding Layer}
\rightarrow
\text{Positional Encoding}
\rightarrow
\text{RMSNorm}
\rightarrow
\text{Multi-Head Attention}
\rightarrow
\text{Output Projection}
\rightarrow
\text{Residual Connection}
\rightarrow
\text{RMSNorm}
\rightarrow
\text{Feed Forward Network}
\rightarrow
\text{Residual Connection}
\rightarrow
\text{Transformer Block Output}
$$

# Complete Transformer Block

## Pre-Context

The Transformer architecture is built by repeating a single computational unit called the **Transformer Block**. Each block receives token representations as input, enriches them using the Self-Attention mechanism, refines them through a Feed Forward Network, and passes the updated representations to the next block.

Modern Large Language Models stack dozens of these Transformer blocks to progressively learn richer semantic and contextual representations.

## What is a Transformer Block?

A Transformer Block is the fundamental building block of every Transformer-based Large Language Model.

Each block consists of the following components:

1. RMSNorm (or LayerNorm)
2. Multi-Head Self-Attention
3. Output Projection
4. Residual Connection
5. RMSNorm (or LayerNorm)
6. Feed Forward Network (FFN)
7. Residual Connection

Together, these components transform the input embeddings into richer contextual representations.

## Transformer Block Architecture

The computation inside one Transformer block is performed as follows.

### Step 1: Normalize the Input

$$
X_1=RMSNorm(X)
$$

### Step 2: Generate Query, Key and Value

$$
Q=X_1W_Q
$$

$$
K=X_1W_K
$$

$$
V=X_1W_V
$$

### Step 3: Compute Multi-Head Attention

$$
Attention(Q,K,V)
=
Softmax
\left(
\frac{QK^T}{\sqrt{d_k}}
\right)V
$$

### Step 4: Output Projection

$$
O=Concat(head_1,\ldots,head_h)W_O
$$

### Step 5: First Residual Connection

$$
Y=X+O
$$

### Step 6: Second Normalization

$$
Y_1=RMSNorm(Y)
$$

### Step 7: Feed Forward Network

$$
F=FFN(Y_1)
$$

### Step 8: Second Residual Connection

$$
Z=Y+F
$$

The matrix

$$
Z
$$

is the final output of one Transformer block.

## Why are Transformer Blocks Stacked?

A single Transformer block can only learn limited contextual information.

By stacking multiple Transformer blocks, the model gradually learns:

- Local relationships
- Long-range dependencies
- Syntax
- Semantics
- Reasoning patterns
- World knowledge

Each block builds upon the representations learned by the previous block.

## Modern Method

Modern decoder-only Large Language Models stack many Transformer blocks.

Examples:

| Model | Number of Transformer Blocks |
|--------|------------------------------:|
| GPT-2 Small | 12 |
| GPT-3 175B | 96 |
| Llama 2 7B | 32 |
| Llama 3 8B | 32 |
| Mistral 7B | 32 |
| Gemma 7B | 28 |
| Qwen 2.5 7B | 28 |

## Libraries and Frameworks

Common implementations include:

- **PyTorch**
- **TensorFlow**
- **JAX**
- **Hugging Face Transformers**

Example:

```python
from transformers.models.llama.modeling_llama import LlamaDecoderLayer
```

Each `LlamaDecoderLayer` represents one complete Transformer block.

## Pipeline

$$
\text{Input}
\rightarrow
\text{RMSNorm}
\rightarrow
\text{Multi-Head Attention}
\rightarrow
\text{Output Projection}
\rightarrow
\text{Residual Connection}
\rightarrow
\text{RMSNorm}
\rightarrow
\text{Feed Forward Network}
\rightarrow
\text{Residual Connection}
\rightarrow
\text{Transformer Block Output}
$$

## Key Takeaways

- A Transformer Block is the fundamental building block of every Transformer-based LLM.
- Each block contains one Multi-Head Attention layer and one Feed Forward Network.
- Two Residual Connections preserve information and improve gradient flow.
- Two normalization layers stabilize training.
- Modern LLMs repeat this block many times to build deep neural networks capable of learning complex language representations.

# Stacking Multiple Transformer Blocks

## Pre-Context

A single Transformer Block can learn contextual relationships between tokens, but its representation is limited. Modern Large Language Models overcome this limitation by stacking multiple Transformer Blocks sequentially. Each block receives the output of the previous block, allowing the model to progressively learn increasingly complex semantic and contextual representations.

## What is Stacking Transformer Blocks?

Stacking Transformer Blocks means placing multiple identical Transformer Blocks one after another.

The output of one block becomes the input to the next block.

If there are \(N\) Transformer Blocks, the computation is:

$$
X_1=Block_1(X)
$$

$$
X_2=Block_2(X_1)
$$

$$
X_3=Block_3(X_2)
$$

$$
\vdots
$$

$$
X_N=Block_N(X_{N-1})
$$

where

- \(X\) is the input embedding.
- \(X_i\) is the output of the \(i^{th}\) Transformer Block.
- \(X_N\) is the final hidden representation.

## Why Stack Multiple Transformer Blocks?

Each Transformer Block learns additional information about the input.

As more blocks are stacked, the model gradually learns:

- Local context
- Long-range dependencies
- Grammar
- Semantics
- Logical reasoning
- World knowledge

Deeper models generally learn richer language representations.

## How Information Flows

The processing pipeline is

$$
Input
\rightarrow
Block_1
\rightarrow
Block_2
\rightarrow
Block_3
\rightarrow
\cdots
\rightarrow
Block_N
$$

Each block refines the hidden representations learned by the previous block.

## Modern Large Language Models

Different LLMs use different numbers of Transformer Blocks.

| Model | Transformer Blocks |
|--------|-------------------:|
| GPT-2 Small | 12 |
| GPT-2 XL | 48 |
| GPT-3 175B | 96 |
| Llama 2 7B | 32 |
| Llama 3 8B | 32 |
| Llama 3 70B | 80 |
| Mistral 7B | 32 |
| Gemma 7B | 28 |
| Qwen 2.5 7B | 28 |

Increasing the number of Transformer Blocks generally improves the model's ability to learn complex patterns, although it also increases computational cost.

## Libraries and Frameworks

Modern implementations stack Transformer Blocks using modules such as:

- **PyTorch**
- **TensorFlow**
- **JAX**
- **Hugging Face Transformers**

Example:

```python
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3-8B"
)
```

Internally, the model consists of multiple Transformer Blocks stacked together.

## Example

Suppose the model contains three Transformer Blocks.

The computation becomes

$$
X
\rightarrow
Block_1
\rightarrow
X_1
$$

$$
X_1
\rightarrow
Block_2
\rightarrow
X_2
$$

$$
X_2
\rightarrow
Block_3
\rightarrow
X_3
$$

The final hidden representation

$$
X_3
$$

is passed to the final normalization layer.

## Pipeline

$$
\text{Raw Text}
\rightarrow
\text{Tokenization}
\rightarrow
\text{Token IDs}
\rightarrow
\text{Embedding Layer}
\rightarrow
\text{Positional Encoding}
\rightarrow
\text{Transformer Block}_1
\rightarrow
\text{Transformer Block}_2
\rightarrow
\cdots
\rightarrow
\text{Transformer Block}_N
\rightarrow
\text{Final RMSNorm}
$$

## Key Takeaways

- A Large Language Model is composed of many identical Transformer Blocks.
- The output of one block becomes the input to the next.
- Each block progressively refines the token representations.
- Stacking more blocks enables the model to learn increasingly complex language patterns.
- The output of the final Transformer Block is passed to the Final RMSNorm layer before generating predictions.

# Final Normalization (Final RMSNorm)

## Pre-Context

After the input has passed through all Transformer Blocks, the model has learned rich contextual representations for every token in the sequence. Before predicting the next token, these hidden representations are passed through one final normalization layer. This ensures that the activations remain numerically stable before the vocabulary projection.

## What is Final RMSNorm?

Final RMSNorm is the last normalization layer applied to the output of the final Transformer Block.

If

$$
H
$$

is the output of the last Transformer Block, then the normalized representation is

$$
H_{norm}
=
RMSNorm(H)
$$

This normalized representation is then passed to the language modeling head (LM Head).

## Why is Final RMSNorm Needed?

The final normalization layer provides several benefits:

- Stabilizes the hidden representations.
- Prevents extremely large activation values.
- Improves numerical stability.
- Produces consistent inputs for the vocabulary projection layer.
- Improves prediction quality.

Without this normalization, the logits may become unstable, leading to poor probability estimates.

## Mathematical Formulation

For an input vector

$$
x=[x_1,x_2,\ldots,x_n]
$$

the Root Mean Square (RMS) is

$$
RMS(x)
=
\sqrt{
\frac{1}{n}
\sum_{i=1}^{n}
x_i^2
}
$$

The normalized output is

$$
RMSNorm(x)
=
\gamma
\frac{x}
{RMS(x)+\epsilon}
$$

where

- \(\gamma\) is a learnable scaling parameter.
- \(\epsilon\) is a small constant for numerical stability.

## Modern Method

Most modern decoder-only Large Language Models use **RMSNorm** as the final normalization layer.

Examples:

- Llama
- Mistral
- Gemma
- Qwen
- DeepSeek

Older Transformer models such as GPT-2 and BERT typically use **LayerNorm** instead.

## Libraries and Frameworks

Common implementations include:

- **PyTorch** (`torch.nn.RMSNorm`)
- **TensorFlow**
- **Hugging Face Transformers**

Example:

```python
import torch.nn as nn

final_norm = nn.RMSNorm(4096)

hidden_states = final_norm(hidden_states)
```

## Example

Suppose the output of the final Transformer Block is

$$
H=
\begin{bmatrix}
1.82 & 0.94 & 1.41 & 0.77\\
1.35 & 1.26 & 0.82 & 1.11
\end{bmatrix}
$$

After applying the final normalization,

$$
H_{norm}
=
\begin{bmatrix}
1.04 & 0.54 & 0.81 & 0.44\\
0.92 & 0.86 & 0.56 & 0.76
\end{bmatrix}
$$

The normalized hidden representation is now ready for the vocabulary projection layer.

## Pipeline

$$
\text{Raw Text}
\rightarrow
\text{Tokenization}
\rightarrow
\text{Token IDs}
\rightarrow
\text{Embedding Layer}
\rightarrow
\text{Positional Encoding}
\rightarrow
\text{Transformer Blocks}
\rightarrow
\text{Final RMSNorm}
\rightarrow
\text{Vocabulary Projection (LM Head)}
$$

## Key Takeaways

- Final RMSNorm is applied after the last Transformer Block.
- It stabilizes the final hidden representations.
- It improves numerical stability before vocabulary projection.
- Modern LLMs typically use RMSNorm, while older models often use LayerNorm.
- The normalized output is passed directly to the LM Head for next-token prediction.

# Vocabulary Projection (LM Head)

## Pre-Context

After passing through all Transformer Blocks and the Final RMSNorm layer, each token is represented by a rich contextual embedding. However, these embeddings cannot yet be interpreted as words. The Language Modeling Head (LM Head) projects the final hidden representations into the vocabulary space, producing a score for every token in the vocabulary.

## What is the LM Head?

The Language Modeling Head (LM Head) is a linear projection layer that maps the hidden representation of each token to the vocabulary size.

If the hidden representation is

$$
H
\in
\mathbb{R}^{n \times d_{model}}
$$

then the LM Head computes

$$
Logits = HW_{vocab}
$$

where

- \(H\) is the final hidden representation.
- \(W_{vocab}\) is the vocabulary projection matrix.

The output matrix has shape

$$
Logits
\in
\mathbb{R}^{n \times |V|}
$$

where

- \(n\) is the sequence length.
- \(|V|\) is the vocabulary size.

## Why is the LM Head Needed?

The Transformer produces contextual embeddings, not words.

The LM Head converts these embeddings into vocabulary scores so that the model can predict the next token.

Without the LM Head, the model cannot generate text.

## Mathematical Formulation

The vocabulary projection is computed as

$$
Logits = HW_{vocab}
$$

where

$$
W_{vocab}
\in
\mathbb{R}^{d_{model}\times |V|}
$$

Each column of \(W_{vocab}\) corresponds to one token in the vocabulary.

## Weight Tying

Modern LLMs often share the embedding matrix and the LM Head weights.

This technique is called **Weight Tying**.

Instead of learning two separate matrices,

$$
Embedding
$$

and

$$
W_{vocab}
$$

the model uses

$$
W_{vocab}
=
Embedding^{T}
$$

Weight tying reduces the number of parameters and often improves model performance.

## Modern Method

Nearly all modern Large Language Models use a linear LM Head.

Examples:

- GPT
- Llama
- Gemma
- Mistral
- Qwen
- DeepSeek

Most modern models also employ **Weight Tying**.

## Libraries and Frameworks

Common implementations include:

- **PyTorch** (`torch.nn.Linear`)
- **TensorFlow**
- **Hugging Face Transformers**

Example:

```python
import torch.nn as nn

lm_head = nn.Linear(
    in_features=4096,
    out_features=32000,
    bias=False
)
```

## Example

Suppose the final hidden representation is

$$
H=
\begin{bmatrix}
0.72 & 1.05 & 0.34 & 0.81
\end{bmatrix}
$$

The vocabulary projection computes

$$
Logits=HW_{vocab}
$$

Result:

$$
Logits=
\begin{bmatrix}
8.2 & 3.5 & 1.4 & 6.8 & \cdots
\end{bmatrix}
$$

Each value corresponds to one token in the vocabulary.

These values are called **logits**.

The logits are then passed to the Softmax function to compute probabilities.

## Pipeline

$$
\text{Raw Text}
\rightarrow
\text{Tokenization}
\rightarrow
\text{Token IDs}
\rightarrow
\text{Embedding Layer}
\rightarrow
\text{Positional Encoding}
\rightarrow
\text{Transformer Blocks}
\rightarrow
\text{Final RMSNorm}
\rightarrow
\text{LM Head}
\rightarrow
\text{Logits}
$$

## Key Takeaways

- The LM Head projects hidden representations into the vocabulary space.
- It produces one score (logit) for every token in the vocabulary.
- Modern LLMs usually use a linear projection layer.
- Many models use Weight Tying to share parameters with the embedding layer.
- The logits produced by the LM Head are the input to the Softmax layer.

# Logits

## Pre-Context

After the final hidden representations are projected into the vocabulary space by the Language Modeling Head (LM Head), the model produces a numerical score for every token in the vocabulary. These scores are called **logits**. Logits indicate how likely each token is to be selected, but they are not probabilities. Before the model can choose the next token, these logits must be converted into probabilities using the Softmax function.

## What are Logits?

Logits are the raw, unnormalized output scores produced by the LM Head.

If

$$
H
$$

is the final hidden representation and

$$
W_{vocab}
$$

is the vocabulary projection matrix, then

$$
Logits = HW_{vocab}
$$

Each logit corresponds to one token in the vocabulary.

For a vocabulary containing \(V\) tokens,

$$
Logits
=
[z_1,z_2,\ldots,z_{|V|}]
$$

where

- \(z_1\) is the score for token 1.
- \(z_2\) is the score for token 2.
- \(\vdots\)
- \(z_{|V|}\) is the score for the last vocabulary token.

## Why are Logits Needed?

The Transformer predicts one score for every token in the vocabulary.

These scores indicate how strongly the model prefers each token.

However,

- Logits can be negative.
- Logits can be greater than 1.
- Logits do not sum to 1.

Therefore, they cannot be interpreted as probabilities.

## Mathematical Formulation

Suppose the vocabulary size is

$$
|V|=50000
$$

Then the LM Head produces

$$
Logits
\in
\mathbb{R}^{50000}
$$

Each element represents the score of one vocabulary token.

## Example

Suppose the vocabulary contains five tokens:

| Token | Logit |
|--------|-------:|
| book | 8.2 |
| pen | 3.4 |
| table | 1.1 |
| chair | 5.8 |
| computer | 2.7 |

The output vector is

$$
Logits=
[8.2,\ 3.4,\ 1.1,\ 5.8,\ 2.7]
$$

Notice that

- The values are not between 0 and 1.
- Their sum is not equal to 1.

These are **scores**, not probabilities.

## Modern Method

Every Transformer-based Large Language Model produces logits before computing probabilities.

Examples:

- GPT
- Llama
- Mistral
- Gemma
- Qwen
- DeepSeek

## Libraries and Frameworks

Logits are produced automatically by the final linear layer.

Common implementations include:

- **PyTorch**
- **TensorFlow**
- **Hugging Face Transformers**

Example:

```python
outputs = model(input_ids)

logits = outputs.logits
```

## What Happens Next?

The logits are passed to the Softmax function.

Softmax converts the raw scores into a probability distribution over the vocabulary.

The token with the highest probability is typically selected during text generation.

## Pipeline

$$
\text{Raw Text}
\rightarrow
\text{Tokenization}
\rightarrow
\text{Token IDs}
\rightarrow
\text{Embedding Layer}
\rightarrow
\text{Positional Encoding}
\rightarrow
\text{Transformer Blocks}
\rightarrow
\text{Final RMSNorm}
\rightarrow
\text{LM Head}
\rightarrow
\text{Logits}
\rightarrow
\text{Softmax}
$$

## Key Takeaways

- Logits are the raw output scores of the LM Head.
- One logit is produced for every token in the vocabulary.
- Logits are not probabilities.
- They may contain negative or very large values.
- Softmax converts logits into probabilities for next-token prediction.

# Softmax

## Pre-Context

After the Language Modeling Head produces logits for every token in the vocabulary, these values cannot be interpreted as probabilities because they may be negative, greater than one, or sum to arbitrary values. The Softmax function transforms these logits into a valid probability distribution, allowing the model to estimate the likelihood of each possible next token.

## What is Softmax?

Softmax is a mathematical function that converts a vector of logits into a probability distribution.

For a vector of logits

$$
z=[z_1,z_2,\ldots,z_n]
$$

the probability of the \(i^{th}\) token is

$$
P_i
=
\frac{e^{z_i}}
{\sum_{j=1}^{n}e^{z_j}}
$$

The resulting probabilities satisfy:

$$
0 \le P_i \le 1
$$

and

$$
\sum_{i=1}^{n}P_i=1
$$

## Why is Softmax Needed?

The output of the LM Head consists of raw scores.

Softmax converts these scores into probabilities so that:

- Every value lies between 0 and 1.
- The probabilities sum to 1.
- The model can compare candidate tokens.
- The next token can be selected.

Without Softmax, the model cannot generate meaningful probability estimates.

## Mathematical Formulation

Given

$$
Logits=[z_1,z_2,\ldots,z_n]
$$

Softmax computes

$$
P_i
=
\frac{e^{z_i}}
{\sum_{j=1}^{n}e^{z_j}}
$$

where

- \(P_i\) is the probability of the \(i^{th}\) token.
- \(z_i\) is the corresponding logit.

## Example

Suppose the logits are

$$
Logits=
[8.2,\ 3.4,\ 1.1,\ 5.8]
$$

After applying Softmax:

| Token | Probability |
|--------|------------:|
| book | 0.90 |
| pen | 0.01 |
| table | 0.00 |
| chair | 0.09 |

The probabilities now sum to

$$
1.0
$$

The token **book** has the highest probability.

## Modern Method

Every modern Transformer-based LLM applies Softmax to the logits before selecting the next token.

Examples:

- GPT
- Llama
- Mistral
- Gemma
- Qwen
- DeepSeek

## Libraries and Frameworks

Common implementations include:

- **PyTorch**
- **TensorFlow**
- **JAX**
- **Hugging Face Transformers**

Example:

```python
import torch

probabilities = torch.softmax(logits, dim=-1)
```

## What Happens Next?

After Softmax produces a probability distribution, the model selects the next token using a decoding algorithm.

Common decoding strategies include:

- Greedy Decoding
- Temperature Sampling
- Top-k Sampling
- Top-p (Nucleus) Sampling
- Beam Search

## Pipeline

$$
\text{Raw Text}
\rightarrow
\text{Tokenization}
\rightarrow
\text{Token IDs}
\rightarrow
\text{Embedding Layer}
\rightarrow
\text{Positional Encoding}
\rightarrow
\text{Transformer Blocks}
\rightarrow
\text{Final RMSNorm}
\rightarrow
\text{LM Head}
\rightarrow
\text{Logits}
\rightarrow
\text{Softmax}
\rightarrow
\text{Decoding}
$$

## Key Takeaways

- Softmax converts logits into probabilities.
- Each probability lies between 0 and 1.
- The probabilities always sum to 1.
- Tokens with larger logits receive higher probabilities.
- The probability distribution is used by the decoding algorithm to generate the next token.

# Decoding Strategies

## Pre-Context

After the Softmax layer converts the logits into probabilities, the model knows the likelihood of every token in the vocabulary. However, it still needs to decide which token to generate next. This decision is performed by a **decoding strategy**. Different decoding methods produce different styles of text, ranging from deterministic outputs to highly creative responses.

## What is Decoding?

Decoding is the process of selecting the next token from the probability distribution produced by the Softmax layer.

Instead of always selecting the token with the highest probability, different decoding algorithms choose tokens in different ways to balance accuracy, diversity, and creativity.

## Why is Decoding Needed?

The Softmax layer only produces probabilities.

A decoding algorithm is required to:

- Select the next token.
- Control randomness.
- Improve response diversity.
- Prevent repetitive outputs.
- Generate fluent and coherent text.

Without decoding, the model cannot generate text.

## Types of Decoding

### 1. Greedy Decoding

Greedy Decoding always selects the token with the highest probability.

Example:

| Token | Probability |
|--------|------------:|
| book | 0.75 |
| pen | 0.12 |
| table | 0.08 |
| chair | 0.05 |

Prediction:

$$
\text{book}
$$

**Advantages**

- Fast
- Deterministic
- Simple

**Disadvantages**

- Can generate repetitive text.
- May miss better long-term sequences.



### 2. Beam Search

Beam Search keeps the top \(k\) most probable sequences instead of only one.

Example:

For

$$
k=3
$$

the algorithm keeps the three best candidate sequences at every generation step.

**Advantages**

- Better sequence quality.
- Finds globally better outputs.

**Disadvantages**

- Computationally expensive.
- Less diverse.



### 3. Temperature Sampling

Temperature controls the randomness of the probability distribution.

The modified probability is computed as

$$
P_i
=
\frac{e^{z_i/T}}
{\sum_j e^{z_j/T}}
$$

where

- \(T<1\) produces more deterministic outputs.
- \(T>1\) produces more diverse outputs.

Typical values:

$$
0.7 \le T \le 1.2
$$



### 4. Top-k Sampling

Instead of considering every vocabulary token, the model keeps only the top \(k\) highest-probability tokens.

Example:

If

$$
k=5
$$

only the five most probable tokens are considered.

The next token is randomly sampled from these candidates.

**Advantages**

- More diverse than Greedy.
- Eliminates extremely unlikely tokens.



### 5. Top-p (Nucleus) Sampling

Instead of selecting a fixed number of tokens, Top-p chooses the smallest set of tokens whose cumulative probability exceeds a threshold \(p\).

Example:

$$
p=0.9
$$

The model keeps adding tokens until the cumulative probability reaches 90%.

Sampling is then performed only within this subset.

Top-p adapts automatically to different probability distributions.



### 6. Contrastive Search

Contrastive Search balances probability with diversity.

Instead of selecting only the highest-probability token, it also penalizes tokens that are too similar to previously generated tokens.

This reduces repetitive text while maintaining fluency.

Modern research models often use Contrastive Search for higher-quality generation.

## Modern Methods

Different applications use different decoding strategies.

| Method | Common Usage |
|---------|--------------|
| Greedy Decoding | Classification |
| Beam Search | Machine Translation |
| Temperature Sampling | Chatbots |
| Top-k Sampling | Text Generation |
| Top-p Sampling | Modern LLMs |
| Contrastive Search | Research Models |

Most modern chat-based LLMs primarily use:

- Temperature Sampling
- Top-p Sampling

## Libraries and Frameworks

Common implementations include:

- **PyTorch**
- **Hugging Face Transformers**
- **vLLM**
- **TensorRT-LLM**

Example:

```python
outputs = model.generate(
    input_ids,
    temperature=0.8,
    top_p=0.9,
    top_k=50
)
```

## Example

Suppose Softmax produces

| Token | Probability |
|--------|------------:|
| book | 0.62 |
| pen | 0.18 |
| table | 0.10 |
| chair | 0.07 |
| phone | 0.03 |

Using

- Greedy → **book**
- Top-k → Randomly sample among the top \(k\) tokens
- Top-p → Sample from the smallest set whose cumulative probability exceeds \(p\)
- Temperature → Adjust randomness before sampling

The selected token is appended to the sequence.

The entire Transformer pipeline is then executed again to predict the next token.

## Pipeline

$$
\text{Raw Text}
\rightarrow
\text{Tokenization}
\rightarrow
\text{Token IDs}
\rightarrow
\text{Embedding Layer}
\rightarrow
\text{Positional Encoding}
\rightarrow
\text{Transformer Blocks}
\rightarrow
\text{Final RMSNorm}
\rightarrow
\text{LM Head}
\rightarrow
\text{Logits}
\rightarrow
\text{Softmax}
\rightarrow
\text{Decoding}
\rightarrow
\text{Next Token}
$$

## Key Takeaways

- Decoding selects the next token from the Softmax probability distribution.
- Different decoding methods produce different text generation behaviors.
- Greedy Decoding is deterministic.
- Beam Search optimizes sequence quality.
- Temperature controls randomness.
- Top-k and Top-p improve diversity.
- Modern LLMs commonly use **Temperature Sampling** together with **Top-p Sampling** for text generation.

# Autoregressive Next Token Generation

## Pre-Context

After the decoding algorithm selects the next token, the generation process does not stop. The predicted token is appended to the original input sequence and fed back into the Transformer. The model then predicts the next token again. This process repeats until an end-of-sequence token is generated or the maximum output length is reached.

## What is Autoregressive Generation?

Autoregressive Generation is the process of generating text one token at a time, where each newly generated token becomes part of the input for predicting the next token.

Instead of generating an entire sentence simultaneously, the Transformer predicts one token during each forward pass.

## Why is Autoregressive Generation Needed?

Language is inherently sequential.

The next word depends on all previously generated words.

Autoregressive generation allows the model to:

- Maintain context.
- Generate coherent sentences.
- Produce variable-length outputs.
- Continue generation until completion.

This is the fundamental generation mechanism used by decoder-only Large Language Models.

## Generation Process

Suppose the input prompt is

```text
Alice gave Bob a
```

The Transformer predicts

```text
book
```

The new input becomes

```text
Alice gave Bob a book
```

The model performs another forward pass and predicts

```text
.
```

The input now becomes

```text
Alice gave Bob a book.
```

The process continues until an End-of-Sequence (EOS) token is generated.

## Mathematical Formulation

Given an input sequence

$$
x_1,x_2,\ldots,x_n
$$

the Transformer predicts

$$
P(x_{n+1}\mid x_1,x_2,\ldots,x_n)
$$

After selecting

$$
x_{n+1}
$$

the sequence becomes

$$
x_1,x_2,\ldots,x_n,x_{n+1}
$$

The model then predicts

$$
P(x_{n+2}\mid x_1,x_2,\ldots,x_{n+1})
$$

This process repeats until generation terminates.

## Stopping Criteria

Text generation stops when one of the following conditions is met:

- End-of-Sequence (EOS) token is generated.
- Maximum token limit is reached.
- User-defined stopping sequence is encountered.

## Modern Method

All modern decoder-only LLMs use autoregressive generation.

Examples:

- GPT
- Llama
- Mistral
- Gemma
- Qwen
- DeepSeek

To improve efficiency, these models use **KV Cache**, which avoids recomputing attention for previously generated tokens.

## Libraries and Frameworks

Common implementations include:

- **Hugging Face Transformers**
- **PyTorch**
- **vLLM**
- **TensorRT-LLM**
- **llama.cpp**

Example:

```python
output = model.generate(
    input_ids,
    max_new_tokens=100
)
```

## Example

Input:

```text
The capital of France is
```

Iteration 1:

```text
Paris
```

Iteration 2:

```text
.
```

Iteration 3:

```text
<EOS>
```

The final generated sentence is

```text
The capital of France is Paris.
```

## Complete LLM Pipeline

$$
\text{Raw Text}
\rightarrow
\text{Tokenization}
\rightarrow
\text{Token IDs}
\rightarrow
\text{Embedding Layer}
\rightarrow
\text{Positional Encoding}
\rightarrow
\text{Transformer Blocks}
\rightarrow
\text{Final RMSNorm}
\rightarrow
\text{LM Head}
\rightarrow
\text{Logits}
\rightarrow
\text{Softmax}
\rightarrow
\text{Decoding}
\rightarrow
\text{Next Token}
\rightarrow
\text{Append Token}
\rightarrow
\text{Repeat Until EOS}
$$

## Key Takeaways

- Decoder-only LLMs generate one token at a time.
- Each predicted token becomes part of the next input.
- The Transformer performs a new forward pass for every generated token.
- Generation stops when an EOS token or another stopping condition is reached.
- Modern LLMs accelerate this process using **KV Cache**, avoiding repeated computation for previous tokens.